<div align="center">

  <h1><img src="https://raw.githubusercontent.com/auxeno/ion/main/assets/logo.png" alt="Ion" width="72"><br>Exploring State Space Models</h1>

  [![Open in nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/Auxeno/ion/blob/main/examples/ssm_pathfinder.ipynb)
  [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/auxeno/ion/blob/main/examples/ssm_pathfinder.ipynb)

</div>

---

SSMs are closely related to RNNs. Both process sequences step-by-step through a hidden state. The key structural difference is *how* that state evolves.

An RNN (like LSTM) uses nonlinear gating to control information flow: sigmoid gates decide what to remember and forget at each step. This is flexible but inherently sequential, with each step depending on the last, giving O(n) parallel time complexity. The gating mechanism is well-suited to discrete, token-level decisions (language modeling, short-to-medium sequences), but the forget gate causes exponential decay that gradually erases information over long horizons.

An SSM replaces gating with a linear recurrence: $h_t = A h_{t-1} + B x_t$. This seems less expressive, but the linearity unlocks two things:

- Parallel training. The linear recurrence can be computed via parallel scan in O(log n) time, matching convolutional and attention-based models.
- Structured dynamics. The matrix $A$ is parameterized with complex eigenvalues that create learnable resonances: oscillatory modes at specific frequencies and decay rates. This gives SSMs a natural inductive bias for continuous signals, spectral structure, and long-range dependencies.

Some variants (S4D, S5) define their dynamics in continuous time and discretize with a learnable timestep $\Delta t$, enabling them to generalize across sampling rates without retraining.

In this notebook we explore three SSM variants in Ion (LRU, S4D, and S5) through hands-on experiments that build intuition for what makes each one tick.

In [1]:
# !pip install ion-nn optax plotly tqdm

In [2]:
import jax
import jax.numpy as jnp
import numpy as np
import optax
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from tqdm import tqdm

import ion
from ion import nn

pio.renderers.default = "notebook_connected"

## What does an SSM actually do?

At each step the hidden state is updated and projected to an output:

$$h_t = \bar{A}\, h_{t-1} + \bar{B}\, x_t, \qquad y_t = C\, h_t + D\, x_t$$

$\bar{A}$ and $\bar{B}$ are the discretized dynamics (derived from continuous parameters $A$, $B$, and timestep $\Delta t$). Rather than deriving them, let's poke an SSM and watch what happens.

In [3]:
# Create an S4D cell and feed it a single pulse
cell = nn.S4DCell(in_dim=1, state_dim=6, key=jax.random.key(0))

# Step through 300 timesteps: input is 1.0 at t=0, then 0.0
h = cell.initial_state
outputs = []
for t in range(300):
    x_in = jnp.ones(1) if t == 0 else jnp.zeros(1)
    x_out, h = cell(x_in, h)
    outputs.append(float(x_out[0]))

fig = go.Figure()
fig.add_trace(go.Scatter(y=outputs, mode="lines", line=dict(width=3, color="#636EFA")))
fig.update_layout(
    title="Impulse response of an S4D cell",
    xaxis_title="Time step",
    yaxis_title="Output",
    height=350,
    template="plotly_white",
    margin=dict(l=60, r=20, t=50, b=50),
)
fig.show()

A single pulse in, and the SSM *rings*: a decaying waveform with multiple frequencies layered together. Where does this come from?

Each of the hidden dimensions contributes one mode: a damped oscillation at its own frequency and decay rate. The full output is their superposition, like harmonics of an instrument.

In [4]:
# Run the same impulse, recording each hidden dimension's contribution
dt = jnp.exp(cell.log_dt)
A = -jnp.exp(cell.A_log_re) + 1j * cell.A_im
A_bar = jnp.exp(A * dt[:, None])
B_bar = (A_bar - 1.0) / A
C = jnp.asarray(cell.C)

h = cell.initial_state
modes = []  # per-mode contributions at each timestep

for t in range(300):
    x_in = jnp.ones(1) if t == 0 else jnp.zeros(1)
    h = A_bar * h + B_bar * x_in[:, None].astype(jnp.complex64)
    mode_contrib = 2.0 * jnp.real(C[0, :] * h[0, :])  # contribution of each mode for feature 0
    modes.append(mode_contrib)

modes = jnp.stack(modes)  # (300, 3)
total = modes.sum(axis=1)  # (300,)

# Consistent colors for each mode
n_modes = modes.shape[1]
mode_colors = ["#636EFA", "#EF553B", "#00CC96"]

fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4],
                    subplot_titles=["Individual modes", "Their sum (the output)"],
                    horizontal_spacing=0.1)

for n in range(n_modes):
    fig.add_trace(go.Scatter(
        y=np.array(modes[:, n]), mode="lines",
        line=dict(width=2, color=mode_colors[n]),
        name=f"Mode {n}", showlegend=False,
    ), row=1, col=1)

fig.add_trace(go.Scatter(
    y=np.array(total), mode="lines",
    line=dict(width=4, color="black"),
    showlegend=False,
), row=1, col=2)

fig.update_layout(height=350, template="plotly_white", margin=dict(l=60, r=20, t=40, b=50))
fig.update_xaxes(title_text="Time step", row=1, col=1)
fig.update_xaxes(title_text="Time step", row=1, col=2)
fig.show()

Each mode is fully determined by one eigenvalue of $\bar{A}$ on the complex plane. For diagonal SSMs like S4D, $\bar{A}$ is a diagonal matrix of complex numbers. Each diagonal entry is one eigenvalue, and each eigenvalue produces one mode.

In [5]:
# Plot eigenvalues alongside their corresponding modes
eigs = A_bar[0, :]  # complex eigenvalues for feature 0

theta = np.linspace(0, 2 * np.pi, 100)

fig = make_subplots(
    rows=1, cols=2, column_widths=[0.4, 0.6],
    subplot_titles=["Eigenvalues on the complex plane", "Corresponding impulse response modes"],
    horizontal_spacing=0.1,
)

# Unit circle
fig.add_trace(go.Scatter(
    x=np.cos(theta), y=np.sin(theta), mode="lines",
    line=dict(dash="dash", color="lightgray", width=1), showlegend=False,
), row=1, col=1)

# Eigenvalue dots with powers (lambda, lambda^2, lambda^3) showing spiral trajectories
power_opacities = [1.0, 0.75, 0.5]

for n in range(n_modes):
    for p, alpha in zip([1, 2, 3], power_opacities):
        eig_p = eigs[n] ** p
        re_p, im_p = float(jnp.real(eig_p)), float(jnp.imag(eig_p))

        # Line from origin to each power
        fig.add_trace(go.Scatter(
            x=[0, re_p], y=[0, im_p], mode="lines",
            line=dict(width=1, color=mode_colors[n]), opacity=alpha, showlegend=False,
        ), row=1, col=1)

        fig.add_trace(go.Scatter(
            x=[re_p], y=[im_p], mode="markers",
            marker=dict(size=12 if p == 1 else 8, color=mode_colors[n]),
            opacity=alpha, showlegend=False,
        ), row=1, col=1)

    fig.add_trace(go.Scatter(
        y=np.array(modes[:, n]), mode="lines",
        line=dict(width=2, color=mode_colors[n]), showlegend=False,
    ), row=1, col=2)

fig.update_xaxes(title_text="Re", range=[-1.15, 1.15], row=1, col=1)
fig.update_yaxes(title_text="Im", range=[-1.15, 1.15], scaleanchor="x", scaleratio=1, row=1, col=1)
fig.update_xaxes(title_text="Time step", row=1, col=2)
fig.update_layout(height=400, template="plotly_white", margin=dict(l=60, r=20, t=40, b=50))
fig.show()

An eigenvalue's distance from the unit circle edge controls how fast its mode decays. Dots near the edge ring for a long time, dots near the center die out quickly. The angle sets the oscillation frequency. The eigenvalues are the SSM's recipe.

The SSM's initial behavior is shaped entirely by these eigenvalues, which come from the initialization scheme. Let's see how different initializations place eigenvalues on the complex plane, and how those placements shape the impulse response before any training.

In [6]:
configs = [
    ("Short memory (r < 0.5)", dict(r_min=0.0, r_max=0.5)),
    ("Medium memory (r < 0.9)", dict(r_min=0.5, r_max=0.9)),
    ("Long memory (r < 0.999)", dict(r_min=0.9, r_max=0.999)),
]

fig = make_subplots(
    rows=3, cols=2, column_widths=[0.35, 0.65],
    horizontal_spacing=0.08, vertical_spacing=0.12,
    subplot_titles=[val for name, _ in configs for val in [name, ""]],
)

theta = np.linspace(0, 2 * np.pi, 100)
row_colors = ["#EF553B", "#FFA15A", "#636EFA"]

for row, ((name, kwargs), color) in enumerate(zip(configs, row_colors)):
    lru_cell = nn.LRUCell(in_dim=1, hidden_dim=8, **kwargs, key=jax.random.key(row))

    # Eigenvalues
    eigs = jnp.exp(-jnp.exp(lru_cell.nu_log) + 1j * jnp.exp(lru_cell.theta_log))

    # Unit circle
    fig.add_trace(go.Scatter(
        x=np.cos(theta), y=np.sin(theta), mode="lines",
        line=dict(dash="dash", color="lightgray", width=1), showlegend=False,
    ), row=row + 1, col=1)

    # Eigenvalue dots
    fig.add_trace(go.Scatter(
        x=np.array(jnp.real(eigs)), y=np.array(jnp.imag(eigs)),
        mode="markers", marker=dict(size=8, color=color), showlegend=False,
    ), row=row + 1, col=1)

    # Impulse response
    h = lru_cell.initial_state
    outputs = []
    for t in range(300):
        x_in = jnp.ones(1) if t == 0 else jnp.zeros(1)
        x_out, h = lru_cell(x_in, h)
        outputs.append(float(x_out[0]))

    fig.add_trace(go.Scatter(
        y=outputs, mode="lines", line=dict(width=3, color=color), showlegend=False,
    ), row=row + 1, col=2)

    fig.update_xaxes(range=[-1.15, 1.15], row=row + 1, col=1)
    fig.update_yaxes(range=[-1.15, 1.15], scaleanchor=f"x{2 * row + 1}", scaleratio=1, row=row + 1, col=1)
    fig.update_xaxes(title_text="Time step" if row == 2 else "", row=row + 1, col=2)

fig.update_layout(height=700, template="plotly_white", margin=dict(l=60, r=20, t=40, b=50))
fig.show()

## Learned spectral decomposition

S4D initializes its eigenvalues at evenly-spaced harmonics: $A_n = -\tfrac{1}{2} + i\pi n$. These create a bank of oscillators at increasing frequencies. What if we train it on signals with specific frequency content? Will it learn to tune its eigenvalues to the right frequencies?

Let's find out by training an S4D to separate a mixture of sinusoids into its individual components.

In [7]:
# Target frequencies to separate
FREQS = jnp.array([3.0, 8.0, 19.0])  # Hz
SAMPLE_RATE = 100.0
DURATION = 2.0
t = jnp.arange(int(SAMPLE_RATE * DURATION)) / SAMPLE_RATE  # (200,)

def generate_signals(key, n_samples):
    key_amp, key_phase, key_noise = jax.random.split(key, 3)
    amps = jax.random.uniform(key_amp, (n_samples, 3), minval=0.3, maxval=1.5)
    phases = jax.random.uniform(key_phase, (n_samples, 3), minval=0.0, maxval=2 * jnp.pi)

    # Individual components: (n_samples, 200)
    components = amps[:, :, None] * jnp.sin(
        2 * jnp.pi * FREQS[None, :, None] * t[None, None, :] + phases[:, :, None]
    )  # (n_samples, 3, 200)

    composite = components.sum(axis=1)  # (n_samples, 200)
    noise = 0.15 * jax.random.normal(key_noise, composite.shape)

    # inputs: (n, 200, 1), targets: (n, 200, 3)
    inputs = (composite + noise)[:, :, None]
    targets = jnp.moveaxis(components, 1, 2)  # (n, 200, 3)
    return inputs, targets

x_train, y_train = generate_signals(jax.random.key(42), 1000)
x_test, y_test = generate_signals(jax.random.key(99), 200)

# Show one example
colors = ["#636EFA", "#EF553B", "#00CC96"]
freq_labels = [f"{f:.0f} Hz" for f in FREQS]

fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=["Input (noisy mixture)"] + freq_labels)

fig.add_trace(go.Scatter(
    x=np.array(t), y=np.array(x_train[0, :, 0]), mode="lines",
    line=dict(width=2, color="gray"), showlegend=False,
), row=1, col=1)

for j in range(3):
    fig.add_trace(go.Scatter(
        x=np.array(t), y=np.array(y_train[0, :, j]), mode="lines",
        line=dict(width=2, color=colors[j]), showlegend=False,
    ), row=j + 2, col=1)

fig.update_layout(height=500, template="plotly_white", margin=dict(l=60, r=20, t=40, b=50))
fig.update_xaxes(title_text="Time (s)", row=4, col=1)
fig.show()

Given the noisy mixture on top, can an S4D learn to isolate each frequency component?

In [8]:
class SignalSeparator(nn.Module):
    expand: nn.Linear
    ssm: nn.S4D
    readout: nn.Linear

    def __init__(self, *, key):
        key_expand, key_ssm, key_readout = jax.random.split(key, 3)
        self.expand = nn.Linear(1, 16, key=key_expand)
        self.ssm = nn.S4D(16, 16, key=key_ssm)
        self.readout = nn.Linear(16, 3, key=key_readout)

    def __call__(self, x):
        x = self.expand(x)      # (b, t, 8)
        x, _ = self.ssm(x)      # (b, t, 8)
        return self.readout(x)  # (b, t, 3)


def sep_loss(model, x, y):
    return jnp.mean((model(x) - y) ** 2)


@jax.jit
def sep_train_step(model, opt, x, y):
    loss, grads = jax.value_and_grad(sep_loss)(model, x, y)
    model, opt = opt.update(model, grads)
    return model, opt, loss


# Save initial eigenvalues before training (feature 0 only for clean visualization)
model_sep = SignalSeparator(key=jax.random.key(0))
A_init = -jnp.exp(model_sep.ssm.cell.A_log_re._value) + 1j * model_sep.ssm.cell.A_im._value
dt_init = jnp.exp(model_sep.ssm.cell.log_dt._value)
eigs_init = jnp.exp(A_init[0, :] * dt_init[0])  # (state_dim//2,) for feature 0

opt_sep = ion.Optimizer(optax.adam(5e-2), model_sep)

losses = []
for epoch in tqdm(range(200), desc="Training"):
    model_sep, opt_sep, loss = sep_train_step(model_sep, opt_sep, x_train, y_train)
    losses.append(float(loss))

fig = go.Figure()
fig.add_trace(go.Scatter(y=losses, mode="lines", line=dict(width=3, color="#636EFA")))
fig.update_layout(
    title="Training loss", xaxis_title="Epoch", yaxis_title="MSE",
    height=300, template="plotly_white", yaxis_type="log",
    margin=dict(l=60, r=20, t=50, b=50),
)
fig.show()

Training: 100%|██████████| 200/200 [01:16<00:00,  2.61it/s]


In [9]:
# Test on one unseen signal
pred = model_sep(x_test[:1])

fig = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    subplot_titles=["Input (noisy mixture)", "3 Hz", "8 Hz", "19 Hz"],
)

fig.add_trace(go.Scatter(
    x=np.array(t), y=np.array(x_test[0, :, 0]), mode="lines",
    line=dict(width=2, color="gray"), showlegend=False,
), row=1, col=1)

for j in range(3):
    fig.add_trace(go.Scatter(
        x=np.array(t), y=np.array(y_test[0, :, j]), mode="lines",
        line=dict(width=2, color=colors[j], dash="dash"), opacity=0.5,
        showlegend=False,
    ), row=j + 2, col=1)
    fig.add_trace(go.Scatter(
        x=np.array(t), y=np.array(pred[0, :, j]), mode="lines",
        line=dict(width=2.5, color=colors[j]), showlegend=False,
    ), row=j + 2, col=1)

fig.update_layout(height=500, template="plotly_white", margin=dict(l=60, r=20, t=40, b=50))
fig.update_xaxes(title_text="Time (s)", row=4, col=1)
fig.show()

Solid lines are the model's predictions, dashed are ground truth. The S4D learned to isolate each component.

Because S4D defines its dynamics in continuous time and discretizes with a learnable timestep $\Delta t$, the same trained model can process signals sampled at any rate. Just scale $\Delta t$ to match the new sampling interval, no retraining needed.

In [10]:
# A continuous signal sampled at different rates
base_rate = 100.0  # Hz
t_ref = np.linspace(0, 2.0, 2000)  # fine reference grid

def continuous_signal(t):
    return np.sin(2 * np.pi * 3.13 * t) + 0.7 * np.sin(2 * np.pi * 7.37 * t) + 0.4 * np.sin(2 * np.pi * 11.91 * t)

signal_ref = continuous_signal(t_ref)

rates = {"10 Hz (0.1x)": 0.1, "100 Hz (1x, training rate)": 1.0, "1000 Hz (10x)": 10.0}

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=list(rates.keys()))

for i, (name, scale) in enumerate(rates.items()):
    actual_rate = base_rate * scale
    t_s = np.arange(0, 2.0, 1.0 / actual_rate)
    sig_s = continuous_signal(t_s)

    fig.add_trace(go.Scatter(
        x=t_ref, y=signal_ref, mode="lines",
        line=dict(color="lightgray", width=1), showlegend=False,
    ), row=i + 1, col=1)
    fig.add_trace(go.Scatter(
        x=t_s, y=sig_s, mode="lines+markers",
        marker=dict(size=3), line=dict(width=2, color="#636EFA"), showlegend=False,
    ), row=i + 1, col=1)

fig.update_layout(height=450, template="plotly_white", margin=dict(l=60, r=20, t=40, b=50))
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.show()

Same signal, three sampling rates. A continuous-time SSM trained at 100 Hz can process any of these by scaling its $\Delta t$. The underlying dynamics don't change, only how finely they're sampled. Discrete-time models like LRU and LSTM have no equivalent mechanism.

## Long-range reasoning: Pathfinder

Can a model trace a path across an image when it only sees one pixel at a time?

Pathfinder is a binary classification task: given a flattened image, determine whether two marked points are connected by a curve. The path can wander across the entire 32x32 image, making this a long-range dependency problem over 1024 time steps.

In [11]:
def cubic_bezier(p0, p1, p2, p3, n_points=150):
    t = np.linspace(0, 1, n_points)[:, None]
    return (1 - t)**3 * p0 + 3 * (1 - t)**2 * t * p1 + 3 * (1 - t) * t**2 * p2 + t**3 * p3


def draw_curve(img, points, value=0.7):
    for x, y in points:
        xi, yi = int(round(x)), int(round(y))
        if 0 <= xi < img.shape[1] and 0 <= yi < img.shape[0]:
            img[yi, xi] = max(img[yi, xi], value)


def draw_dot(img, center, radius=1, value=1.0):
    cy, cx = int(round(center[1])), int(round(center[0]))
    for dy in range(-radius, radius + 1):
        for dx in range(-radius, radius + 1):
            y, x = cy + dy, cx + dx
            if 0 <= y < img.shape[0] and 0 <= x < img.shape[1]:
                img[y, x] = value


def generate_pathfinder(n_samples, img_size=32, n_distractors=4, seed=0):
    rng = np.random.RandomState(seed)
    images = []
    labels = []

    for i in range(n_samples):
        img = np.zeros((img_size, img_size), dtype=np.float32)

        # Two anchor points in top and bottom thirds
        p1 = np.array([rng.randint(4, img_size - 4), rng.randint(3, img_size // 3)])
        p2 = np.array([rng.randint(4, img_size - 4), rng.randint(2 * img_size // 3, img_size - 3)])

        connected = i < n_samples // 2

        if connected:
            # Bezier curve connecting the two anchors
            mid_y = (p1[1] + p2[1]) / 2
            cp1 = np.array([p1[0] + rng.uniform(-10, 10), mid_y + rng.uniform(-8, 8)])
            cp2 = np.array([p2[0] + rng.uniform(-10, 10), mid_y + rng.uniform(-8, 8)])
            curve = cubic_bezier(p1.astype(float), cp1, cp2, p2.astype(float))
            draw_curve(img, curve, value=0.8)
        else:
            # Two separate short curves near each anchor
            for anchor in [p1, p2]:
                end = anchor + rng.uniform(-6, 6, size=2)
                end[1] = np.clip(end[1], 2, img_size - 3)
                cp1 = anchor + rng.uniform(-5, 5, size=2)
                cp2 = end + rng.uniform(-3, 3, size=2)
                curve = cubic_bezier(anchor.astype(float), cp1, cp2, end)
                draw_curve(img, curve, value=0.8)

        # Distractor curves
        for _ in range(n_distractors):
            s = rng.uniform(2, img_size - 2, size=2)
            e = rng.uniform(2, img_size - 2, size=2)
            c1 = s + rng.uniform(-8, 8, size=2)
            c2 = e + rng.uniform(-8, 8, size=2)
            curve = cubic_bezier(s, c1, c2, e)
            draw_curve(img, curve, value=0.4)

        # Draw anchor dots on top
        draw_dot(img, p1, value=1.0)
        draw_dot(img, p2, value=1.0)

        images.append(img)
        labels.append(1 if connected else 0)

    return np.stack(images), np.array(labels)


train_imgs, train_labels = generate_pathfinder(1000, seed=42)
test_imgs, test_labels = generate_pathfinder(200, seed=99)

# Display examples
fig = make_subplots(rows=2, cols=4, vertical_spacing=0.08, horizontal_spacing=0.04,
                    subplot_titles=["Connected"] * 4 + ["Disconnected"] * 4)

for col in range(4):
    for row, offset in enumerate([0, 500]):
        idx = offset + col
        img_rgb = np.stack([train_imgs[idx]] * 3, axis=-1)
        img_uint8 = (img_rgb * 255).astype(np.uint8)
        fig.add_trace(go.Image(z=img_uint8), row=row + 1, col=col + 1)
        fig.update_xaxes(showticklabels=False, row=row + 1, col=col + 1)
        fig.update_yaxes(showticklabels=False, row=row + 1, col=col + 1)

fig.update_layout(height=350, template="plotly_white", margin=dict(l=10, r=10, t=40, b=10))
fig.show()

In [12]:
# What the model actually sees: a flattened 1024-step sequence
connected_img = train_imgs[0]
disconnected_img = train_imgs[501]

fig = make_subplots(rows=2, cols=2, column_widths=[0.3, 0.7], horizontal_spacing=0.08,
                    vertical_spacing=0.12,
                    subplot_titles=["Connected", "Flattened sequence",
                                    "Disconnected", "Flattened sequence"])

for row, (img, color) in enumerate([(connected_img, "#636EFA"), (disconnected_img, "#EF553B")]):
    seq = img.flatten()
    img_rgb = np.stack([img] * 3, axis=-1)
    fig.add_trace(go.Image(z=(img_rgb * 255).astype(np.uint8)), row=row + 1, col=1)
    fig.update_xaxes(showticklabels=False, row=row + 1, col=1)
    fig.update_yaxes(showticklabels=False, row=row + 1, col=1)

    fig.add_trace(go.Scatter(
        y=seq, mode="lines", line=dict(width=1, color=color), showlegend=False,
    ), row=row + 1, col=2)
    fig.update_yaxes(title_text="Pixel value", row=row + 1, col=2)

fig.update_xaxes(title_text="Pixel index (time step)", row=2, col=2)
fig.update_layout(height=500, template="plotly_white", margin=dict(l=10, r=20, t=40, b=50))
fig.show()

The model sees the 1D sequence on the right and must decide: are the two bright dots connected by a path? The dots can be hundreds of time steps apart in the flattened sequence.

In [13]:
class PathfinderSSM(nn.Module):
    expand: nn.Linear
    ssm: nn.S5
    classify: nn.Linear

    def __init__(self, state_dim=32, *, key):
        key_expand, key_ssm, key_classify = jax.random.split(key, 3)
        self.expand = nn.Linear(1, 32, key=key_expand)
        self.ssm = nn.S5(32, state_dim, key=key_ssm)
        self.classify = nn.Linear(32, 2, key=key_classify)

    def __call__(self, x):
        x = jax.nn.gelu(self.expand(x))
        x, _ = self.ssm(x)
        x = jnp.mean(x, axis=1)  # mean pool over time
        return self.classify(x)


class PathfinderLSTM(nn.Module):
    expand: nn.Linear
    lstm: nn.LSTM
    classify: nn.Linear

    def __init__(self, hidden_dim=32, *, key):
        key_expand, key_lstm, key_classify = jax.random.split(key, 3)
        self.expand = nn.Linear(1, 32, key=key_expand)
        self.lstm = nn.LSTM(32, hidden_dim, key=key_lstm)
        self.classify = nn.Linear(hidden_dim, 2, key=key_classify)

    def __call__(self, x):
        x = jax.nn.gelu(self.expand(x))
        _, (h, c) = self.lstm(x)
        return self.classify(h)


def cls_loss(model, x, labels):
    logits = model(x)
    return optax.softmax_cross_entropy_with_integer_labels(logits, labels).mean()


@jax.jit
def cls_step(model, opt, x, labels):
    loss, grads = jax.value_and_grad(cls_loss)(model, x, labels)
    model, opt = opt.update(model, grads)
    return model, opt, loss


@jax.jit
def cls_accuracy(model, x, labels):
    return (jnp.argmax(model(x), axis=-1) == labels).mean()


# Prepare data: flatten images to (n, 1024, 1)
x_train_seq = jnp.array(train_imgs.reshape(len(train_imgs), -1, 1))
y_train_cls = jnp.array(train_labels)
x_test_seq = jnp.array(test_imgs.reshape(len(test_imgs), -1, 1))
y_test_cls = jnp.array(test_labels)

BATCH_SIZE = 64
n_batches = len(x_train_seq) // BATCH_SIZE

results = {}
for name, cls in [("S5", PathfinderSSM), ("LSTM", PathfinderLSTM)]:
    m = cls(key=jax.random.key(0))
    o = ion.Optimizer(optax.chain(optax.clip_by_global_norm(1.0), optax.adam(1e-3)), m)
    accs = []

    for epoch in tqdm(range(30), desc=name):
        perm = jax.random.permutation(jax.random.key(epoch), len(x_train_seq))
        for i in range(n_batches):
            idx = perm[i * BATCH_SIZE : (i + 1) * BATCH_SIZE]
            m, o, loss = cls_step(m, o, x_train_seq[idx], y_train_cls[idx])

        acc = float(cls_accuracy(m, x_test_seq, y_test_cls))
        accs.append(acc)

    results[name] = accs
    print(f"  {name} final test accuracy: {accs[-1]:.1%}")

S5: 100%|██████████| 30/30 [00:16<00:00,  1.82it/s]


  S5 final test accuracy: 83.0%


LSTM: 100%|██████████| 30/30 [00:43<00:00,  1.44s/it]

  LSTM final test accuracy: 49.0%


In [14]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=results["S5"], mode="lines", name="S5",
    line=dict(width=4, color="#636EFA"), opacity=0.85,
))
fig.add_trace(go.Scatter(
    y=results["LSTM"], mode="lines", name="LSTM",
    line=dict(width=4, color="#EF553B"), opacity=0.85,
))
fig.add_hline(y=0.5, line_dash="dash", line_color="lightgray", annotation_text="Random chance")
fig.update_layout(
    title="Pathfinder: test accuracy over training",
    xaxis_title="Epoch", yaxis_title="Test accuracy",
    height=400, template="plotly_white",
    margin=dict(l=60, r=20, t=50, b=50),
    yaxis=dict(range=[0.4, 1.0]),
)
fig.show()

---

SSMs combine the parallel training efficiency of convolutions with the sequential inference of RNNs, and add a continuous-time inductive bias that neither has. Each variant brings something different:

- LRU: the simplest SSM. Flexible eigenvalue placement on a configurable annulus, no continuous-time assumption. A good baseline and starting point to learn from.
- S4D: per-feature SISO structure with continuous-time dynamics. Each feature has its own independent filter bank, making it natural for spectral tasks.
- S5: MIMO with shared state and dense projections. Features interact inside the state space, making it useful for modelling linked dynamic systems.

For more, see the original papers: [S4](https://arxiv.org/abs/2111.00396) (Gu et al., 2021), [S4D](https://arxiv.org/abs/2206.11893) (Gu et al., 2022), [S5](https://arxiv.org/abs/2208.04933) (Smith et al., 2023), [LRU](https://arxiv.org/abs/2303.06349) (Orvieto et al., 2023).